In [1]:
import os
import openai
import yaml
import zipfile
from functools import lru_cache
import uuid
import json

In [2]:
knwf_path = "/Users/khanh/Repository/imaging_KNIME_to_Galaxy/src/knime2galaxy/data/file_to_translate/2025_02_image_conversion_nested.knwf"


1. Extract a KNIME .knwf file into the target directory.

In [3]:
def collect_knime_node_files(knwf_path: str) -> dict:
    """
    Collects all node settings.xml files inside the KNIME .knwf archive.
    Returns a dictionary: {node_folder_name: xml_content}.
    """
    node_data = {}
    with zipfile.ZipFile(knwf_path, "r") as zf:
        for file_name in zf.namelist():
            if file_name.endswith("settings.xml"):
                with zf.open(file_name) as f:
                    xml_content = f.read().decode("utf-8")
                    node_name = file_name.split("/")[-2]  # Ordnername
                    node_data[node_name] = xml_content
    return node_data

In [4]:
knime_nodes = collect_knime_node_files(knwf_path)
print("First 100 characters in every node:")
for node_name, xml_content in knime_nodes.items():
    print(f"{node_name}: {xml_content[:100]}...")

First 100 characters in every node:
Image Writer (#2): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Reader (#1): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Writer (#3): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Writer (#4): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...


In [5]:
# Collects the workflow.knime file
def collect_workflow_file(knwf_path: str) -> str:
    """
    Extracts the content of the workflow.knime file inside the KNIME .knwf archive.
    Returns the file content as a string.
    """
    with zipfile.ZipFile(knwf_path, "r") as zf:
        for file_name in zf.namelist():
            if file_name.endswith("workflow.knime"):
                with zf.open(file_name) as f:
                    return f.read().decode("utf-8")
    raise FileNotFoundError("workflow.knime not found in KNWF archive")

In [6]:
# Output the first 500 characters of the workflow
workflow_content = collect_workflow_file(knwf_path)
print(workflow_content[:500])  

<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="workflow.knime">
    <entry key="created_by" type="xstring" value="4.7.7.v202308161346"/>
    <entry key="created_by_nightly" type="xboolean" value="false"/>
    <entry key="version" type="xstring" value="4.1.0"/>
    <entry key="name" type="xs


In [10]:

with open("data/translation_table.yml") as f:
    raw_steps = yaml.safe_load(f)

print(f"{len(raw_steps)} Blöcke geladen")
raw_steps[:1]


7 Blöcke geladen


[{'description': 'Input Image dataset',
  'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n',
  'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n",
  'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmln

In [12]:
def step_to_text(step: dict) -> str:
    parts = []
    if step.get("description"):
        parts.append(f"Description: {step['description']}")

    if step.get("Galaxy"):
        parts.append(f"Galaxy:\n{step['Galaxy']}")

    if step.get("Python"):
        parts.append(f"Python:\n{step['Python']}")

    if step.get("KNIME"):
        parts.append(f"KNIME:\n{step['KNIME']}")

    return "\n".join(parts).strip()

corpus_texts = [step_to_text(s) for s in raw_steps]

# Vorschau auf die ersten 500 Zeichen des ersten Blocks
print(corpus_texts[0][:500])


Description: Input Image dataset
Galaxy:
"0": {
        "annotation": "",
        "content_id": null,
        "errors": null,
        "id": 0,
        "input_connections": {},
        "inputs": [
            {
                "description": "",
                "name": "Input Image Dataset"
            }
        ],
        "label": "Input Image Dataset",
        "name": "Input dataset",
        "outputs": [],
        "position": {
            "left": 0.0,
            "top": 0.0
        },
       


## Vector embeddings
To make our code snippets searchable, we need to created vector embedding form them, we need to turn them into vectors.

In [ ]:
from openai import OpenAI
client = OpenAI()

models = client.models.list() 

embedding_models = [m.id for m in models.data if "embedding" in m.id]
print(embedding_models)

['text-embedding-3-small', 'text-embedding-3-large', 'text-embedding-ada-002']


In [15]:
def embed(text):
    client = OpenAI()

    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

## Vector store

We also need a vector store, which is basically just a dictionary that allows us to quickly find a text given a corresponding vector, or a vector that has a short distance to it.

In [ ]:
import numpy as np

class VectorStore:
    def __init__(self, texts=None, use_cosine=True):
        self.texts = []
        self.vectors = []         # Liste np.ndarray (float32)
        self.use_cosine = use_cosine
        if texts:
            self.add_many(texts)

    @staticmethod
    def _normalize(v):
        v = np.asarray(v, dtype=np.float32)
        n = np.linalg.norm(v)
        return v / (n + 1e-12)

    def add(self, text):
        vec = embed(text)
        if self.use_cosine:
            vec = self._normalize(vec)
        else:
            vec = np.asarray(vec, dtype=np.float32)
        self.texts.append(text)
        self.vectors.append(vec)

    def add_many(self, texts):
        for t in texts:
            self.add(t)

    def search(self, query, n_best_results=3):
        if not self.vectors:
            return []

        q = embed(query)
        q = self._normalize(q) if self.use_cosine else np.asarray(q, dtype=np.float32)
        M = np.vstack(self.vectors)              # [N, D]

        if self.use_cosine:
            # Kosinus-Ähnlichkeit (höher = besser)
            scores = M @ q
            idx = np.argsort(-scores)[:n_best_results]
            return [(self.texts[i], float(scores[i])) for i in idx]
        else:
            # Euklidische Distanz (kleiner = besser)
            dists = np.linalg.norm(M - q, axis=1)
            idx = np.argsort(dists)[:n_best_results]
            # optional: in Score umrechnen (1/(1+d))
            scores = 1.0 / (1.0 + dists[idx])
            return [(self.texts[i], float(scores[j])) for j, i in enumerate(idx)]

    def get_text(self, i):
        return self.texts[i]

In [ ]:
store = VectorStore(corpus_texts)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
# def build_examples():
#     examples_text =  """You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

# Below are examples of how this translation should be done:


# """
#     return examples_text


In [ ]:
node_examples = build_examples()
print(node_examples)

In [ ]:
# Convert dict content to string.
knime_nodes_str = "\n".join(
    f"Node ID: {key}\n{value}" for key, value in knime_nodes.items()
)
print(knime_nodes_str)

In [ ]:
def build_task_prompt():
    
    task_prompt = f"""
# Your Task
You are a system that translates complete KNIME workflows into Galaxy workflows. Produce a **single, valid Galaxy .ga workflow JSON** that can be imported in Galaxy, representing the entire KNIME workflow below.



# Output Requirements
- Respond with the complete Galaxy workflow JSON object ONLY (no markdown fences, no comments, no explanations).
- The JSON must be a valid Galaxy .ga workflow 
- Make sure that it is a valid JSON object.
- For uuid fields, write 00000000-0000-0000-0000-000000000000 as placeholder
- Do not include TODOs or comments in the JSON.

"""
    
    return task_prompt

In [ ]:
task = build_task_prompt()
print(task)

In [ ]:
def build_workflow_examples():
    workflow_examples_text =  """
    Here are some examples of complete KNIME workflows and their corresponding Galaxy workflows:
    """  
    examples = load_translation_examples(yaml_path="data/workflow_translation_table.yml")
    if examples:
        for i, ex in enumerate(examples[:]): 
            workflow_examples_text += f"""

## Example {i + 1}:

KNIME node (XML):

```xml
{ex["KNIME"]}
```

Galaxy step (JSON):
```json
{ex["Galaxy"]}
```

"""
    return workflow_examples_text

In [ ]:
workflow_examples = build_workflow_examples()
print(workflow_examples)

In [ ]:
full_prompt = f"{node_examples}\n\n{workflow_examples}\n\n{task}"
print(full_prompt)

In [ ]:
def list_scadsai_models():
    client = openai.OpenAI(
        base_url="https://llm.scads.ai/v1",
        api_key=os.environ.get("SCADSAI_API_KEY")
    )
    models = client.models.list()
    return [m.id for m in models.data]

In [ ]:
list_scadsai_models()

In [ ]:
def prompt_scadsai_llm(message:str, model="openai/gpt-oss-120b"):
    """A prompt helper function that sends a message to ScaDS.AI LLM server at 
    ZIH TU Dresden and returns only the text response.
    """
    
    # convert message in the right format if necessary
    if isinstance(message, str):
        message = [{"role": "user", "content": message}]
    
    # setup connection to the LLM
    client = openai.OpenAI(base_url="https://llm.scads.ai/v1",
                           api_key=os.environ.get('SCADSAI_API_KEY')
    )
    response = client.chat.completions.create(
        model=model,
        messages=message
    )
    
    # extract answer
    return response.choices[0].message.content

In [ ]:
answer = prompt_scadsai_llm(message= full_prompt)
print(answer)

In [ ]:
# Parse the answer to a JSON object

try: 
    json_object = json.loads(answer)
    print("Parsed JSON successfully.")
    print(json_object)
except json.JSONDecodeError as e:
    print("Failed to parse JSON:", e)


In [ ]:
if "uuid" in json_object:
    json_object["uuid"] = str(uuid.uuid4())

In [ ]:
# Go through the JSON object recursively searching for the "uuid" key and replace the correspondign value with a new UUID
# Use the uuid module to generate a new UUID
for step in json_object["steps"].values():
    if isinstance(step, dict) and "uuid" in step:
        step["uuid"] = str(uuid.uuid4())

In [ ]:
print(json_object["uuid"])
print([s["uuid"] for s in json_object["steps"].values()])

In [ ]:
# Save the answer to a file
output_file = "data/output_file/knime2galaxy_workflow_output3.ga"
with open(output_file, "w", encoding="utf-8") as f:
      json.dump(json_object, f, indent=2, ensure_ascii=False)

print(f"Galaxy workflow saved to {output_file}")